# 🗂️ Notebook 2: S3 (Object Storage) — Data Model & APIs

In this notebook we build a tiny S3 in pure Python, iterating from a
**naive bad version** to a **reasonable first cut**. The point isn't
to be production-ready — it's to feel *why* each feature exists by
watching the previous version break.

Progression:

1. **Bad:** a dict. Silently overwrites, no metadata, no integrity.
2. **Better:** versioning, tombstones, ETag, typed metadata.
3. **Best (today):** multipart upload for large files.


## 🛠️ Setup

```bash
cd 06-system-designs/s3
uv sync
```

Then in VS Code pick the `.venv` kernel from the top-right of the notebook. If
it doesn't show up: `Cmd+Shift+P` → **Reload Window** and try again.

Everything in this lab is **pure Python** — no databases, no Docker. You can
run it on a laptop in a few seconds.


## 🧱 Entities

- **Bucket** — a namespace and a policy container.
- **Object** — addressed by `(bucket, key, version_id)`, carries a
  blob of bytes plus some metadata (size, ETag, content-type, …).
- **Shard** — a piece of the blob that lives on one storage node.
  We won't implement shards in this notebook; that's Notebook 3.


## 🌐 HTTP-style API

| Method | Path | What it does |
|---|---|---|
| `PUT`    | `/{bucket}` | Create a bucket |
| `PUT`    | `/{bucket}/{key}` | Upload an object |
| `GET`    | `/{bucket}/{key}?versionId=…` | Download an object (or a specific version) |
| `DELETE` | `/{bucket}/{key}` | Delete — creates a **tombstone** if versioning is on |
| `GET`    | `/{bucket}?list&prefix=…` | List objects with a prefix |
| `POST`   | `/{bucket}/{key}?uploads` | Start a multipart upload |
| `PUT`    | `/{bucket}/{key}?partNumber=i&uploadId=…` | Upload one part |
| `POST`   | `/{bucket}/{key}?uploadId=…` | Complete the multipart upload |

Notice the REST shape: buckets and objects are *resources* with URLs,
the verb comes from HTTP. This keeps the API small and familiar.


## 1️⃣ BAD: a naive dict 🙈

Let's start with the simplest thing: a `dict` keyed by `(bucket,
key)`. No versioning, no metadata, no integrity check.


In [1]:
naive = {}

def bad_put(bucket, key, data):
    naive[(bucket, key)] = data

def bad_get(bucket, key):
    return naive.get((bucket, key))

bad_put("photos", "cat.jpg", b"v1-bytes")
bad_put("photos", "cat.jpg", b"v2-bytes")   # silently overwrites v1!
print("old bytes recoverable? ->", bad_get("photos", "cat.jpg") == b"v1-bytes")
print("there is no record that v1 ever existed.")


old bytes recoverable? -> False
there is no record that v1 ever existed.


**Three concrete problems** with this version:

1. A careless `PUT` erases the previous contents forever.
2. We can't answer "when was this object created?" or "how big is
   it?" without reading the blob.
3. If a disk flips a bit while we're reading, we happily return the
   corrupted bytes.

Let's fix those.


## 2️⃣ BETTER: typed metadata, versioning, ETag 🛡️

We introduce:

- A **pydantic** `ObjectMeta` model — each version gets one.
- **Versioning**: every `PUT` appends a new version; `DELETE` appends
  a **tombstone** (also called a delete marker) instead of erasing.
- **ETag** = MD5 of the bytes. The client and server can both
  compute it and compare — a corrupt read is detectable.


In [2]:
from datetime import datetime, timezone
from hashlib import md5
from pydantic import BaseModel, Field
from collections import defaultdict
import uuid


class ObjectMeta(BaseModel):
    bucket: str
    key: str
    version_id: str
    size: int = Field(ge=0)
    etag: str
    content_type: str = "application/octet-stream"
    is_delete_marker: bool = False
    created_at: datetime


class VersionedStore:
    def __init__(self):
        # (bucket, key) -> list of ObjectMeta, newest last
        self.meta: dict = defaultdict(list)
        # version_id -> raw bytes (the "data plane")
        self.blobs: dict = {}

    def put(self, bucket, key, data, content_type="application/octet-stream"):
        vid = uuid.uuid4().hex[:8]
        m = ObjectMeta(
            bucket=bucket, key=key, version_id=vid,
            size=len(data), etag=md5(data).hexdigest(),
            content_type=content_type,
            created_at=datetime.now(timezone.utc),
        )
        self.meta[(bucket, key)].append(m)
        self.blobs[vid] = data
        return m

    def delete(self, bucket, key):
        # Tombstone: the bytes stay, but the "latest" is now a delete marker.
        vid = uuid.uuid4().hex[:8]
        m = ObjectMeta(
            bucket=bucket, key=key, version_id=vid,
            size=0, etag="", is_delete_marker=True,
            created_at=datetime.now(timezone.utc),
        )
        self.meta[(bucket, key)].append(m)
        return m

    def get(self, bucket, key, version_id=None):
        versions = self.meta[(bucket, key)]
        if version_id is not None:
            for m in versions:
                if m.version_id == version_id:
                    if m.is_delete_marker:
                        return None
                    return m, self.blobs.get(m.version_id)
            return None
        if not versions:
            return None
        m = versions[-1]
        if m.is_delete_marker:
            return None
        return m, self.blobs.get(m.version_id)

    def list(self, bucket, prefix="", limit=1000):
        out = []
        for (b, k), versions in self.meta.items():
            if b != bucket or not k.startswith(prefix):
                continue
            latest = versions[-1]
            if latest.is_delete_marker:
                continue
            out.append(latest)
            if len(out) >= limit:
                break
        return out


In [3]:
# A small tour of VersionedStore
store = VersionedStore()

m1 = store.put("photos", "cat.jpg", b"v1-bytes")
m2 = store.put("photos", "cat.jpg", b"v2-bytes")
store.delete("photos", "cat.jpg")              # <- oops!
store.put("photos", "dog.jpg", b"woof")

print("Latest cat.jpg (after delete):", store.get("photos", "cat.jpg"))

# 🪄 Undelete: just ask for an older version_id.
recovered = store.get("photos", "cat.jpg", m1.version_id)
print("Recovered v1 of cat.jpg:", recovered[1] if recovered else None)

print("List:", [m.key for m in store.list("photos")])


Latest cat.jpg (after delete): None
Recovered v1 of cat.jpg: b'v1-bytes'
List: ['dog.jpg']


A delete is now **reversible** as long as the old versions exist —
this is how S3 bucket versioning + MFA-delete protect you from
ransomware and from your own `rm -rf`.


In [4]:
# ETag catches corruption: a single flipped byte changes the hash.
data = b"important file"
etag = md5(data).hexdigest()
corrupted = data[:-1] + b"?"
print("match?          ", md5(data).hexdigest()      == etag)
print("corrupted match?", md5(corrupted).hexdigest() == etag)


match?           True
corrupted match? False


## 3️⃣ BEST (today): multipart upload 🧩

Imagine uploading a **50 GB** video in one HTTP `PUT`. Two things go
wrong:

1. If your connection drops at 90%, you restart at 0%.
2. You can't parallelize — you're bottlenecked by one TCP connection.

The fix is **multipart upload**: the client chops the file into
chunks (5 MB–5 GB each), uploads them independently (in parallel, in
any order), then sends a "complete" request with the list of parts.
The server stitches them together.

Let's first *feel* the pain of the naive approach, then build the
resumable version.


In [5]:
# BAD: one huge PUT. Every dropped connection restarts from zero.
import random

def bad_upload(size_mb: int, loss_prob_per_mb: float = 0.01) -> int:
    attempts = 0
    while True:
        attempts += 1
        sent = 0
        for _ in range(size_mb):
            if random.random() < loss_prob_per_mb:
                break
            sent += 1
        if sent == size_mb:
            return attempts

random.seed(42)
print("attempts needed for 500 MB single upload:", bad_upload(500))


attempts needed for 500 MB single upload: 67


In [6]:
# BEST: resumable, parallelisable multipart upload.
from hashlib import md5
import uuid

class MultipartUpload:
    def __init__(self, bucket, key, part_size=5 * 1024 * 1024):
        self.bucket, self.key = bucket, key
        self.part_size = part_size
        self.upload_id = uuid.uuid4().hex[:8]
        self.parts: dict = {}           # part_number -> bytes

    def upload_part(self, part_number: int, data: bytes) -> str:
        assert 1 <= part_number <= 10_000, "S3 caps at 10,000 parts"
        self.parts[part_number] = data
        return md5(data).hexdigest()    # per-part ETag

    def complete(self):
        # Parts may arrive out of order — sort by part number.
        ordered = [self.parts[i] for i in sorted(self.parts)]
        blob = b"".join(ordered)
        # Real-S3 multipart ETag: md5 over the concatenation of
        # each part's binary md5, suffixed with "-<numparts>".
        concat = b"".join(md5(p).digest() for p in ordered)
        etag = f"{md5(concat).hexdigest()}-{len(ordered)}"
        return blob, etag


mpu = MultipartUpload("videos", "movie.mp4")
payload = b"A" * (5 * 1024 * 1024) + b"B" * (5 * 1024 * 1024) + b"C" * 1024

mpu.upload_part(1, payload[:5 * 1024 * 1024])
# imagine the network dying here; part 2 just gets retried
mpu.upload_part(2, payload[5 * 1024 * 1024:10 * 1024 * 1024])
mpu.upload_part(3, payload[10 * 1024 * 1024:])

blob, etag = mpu.complete()
print("reconstructed size:", len(blob))
print("multipart ETag    :", etag)


reconstructed size: 10486784
multipart ETag    : 1c3dafcf2149c7b623d2361636f6528c-3


Why this design is lovely:

- **Resumable** — only the failing part retries.
- **Parallel** — the client can push 8 parts at once over 8 TCP
  connections to saturate their pipe.
- **Out-of-order friendly** — parts are keyed by number, not by
  offset, so any ordering works.
- **Self-describing** — the funny "etag-N" format tells any reader
  how many parts the object was uploaded in.


## 🚦 Consistency: one line everybody gets wrong

*After I `PUT` and get a 200, can I `GET` the new bytes immediately?*

- **Same key, same region**: yes. Strong read-after-write for new
  objects and overwrites (since Dec 2020 for real S3).
- **Listing** after a write: **eventual**. The new object may not
  appear in a `LIST` for a few seconds.
- **Cross-region replication**: **eventual**, typically seconds but
  occasionally minutes.

Our toy store always reads the latest metadata in the same process,
so it looks strongly consistent. A real distributed implementation
keeps metadata in a strongly-consistent sharded DB and relies on the
fact that **metadata is small** to make that affordable.


## 🔁 Takeaways

- Bad design: "just a dict" loses data and has no observability.
- Fix: typed metadata (pydantic), versioning + tombstones, ETag for
  integrity.
- For large objects, **multipart upload** is non-negotiable. It also
  maps naturally onto erasure-coded shards (Notebook 3).
- Consistency is *always* a tradeoff statement. Write the statement
  down.
